# DrowsyGuard — huấn luyện CNN mắt

Notebook dùng ảnh mắt MRL đã tải sẵn trong Google Drive; không tải dataset. Đặt toàn bộ project tại `MyDrive/DrowsyGuard` để notebook dùng `ml/colab_training.py` và xuất model vào `models/`. Ảnh MRL thường đã là crop mắt; tên file chuẩn mã hóa subject ở field đầu và trạng thái mắt ở field thứ tư (index 3). Chỉnh parser nếu bản dataset của bạn khác.

Nhãn: `0 = mắt mở`, `1 = mắt nhắm`. Tập test được tách theo subject, không theo ảnh.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys
PROJECT_ROOT = Path('/content/drive/MyDrive/DrowsyGuard')
if not (PROJECT_ROOT / 'ml' / 'colab_training.py').is_file():
    raise FileNotFoundError('Hãy đặt/copy project DrowsyGuard vào MyDrive/DrowsyGuard.')
sys.path.insert(0, str(PROJECT_ROOT))
from ml.colab_training import (
    make_tf_dataset, report_binary_metrics, scan_mrl_eye_dataset,
    split_by_subject, train_mobilenet_binary, export_tflite,
)

In [ ]:
# Chỉnh đường dẫn sau khi giải nén dataset trong Drive.
DATASET_ROOT = Path('/content/drive/MyDrive/datasets/MRL_Eye')
MRL_STATE_FIELD_INDEX = 3
MRL_CLOSED_VALUE = '0'
MRL_OPEN_VALUE = '1'

records = scan_mrl_eye_dataset(
    DATASET_ROOT,
    state_field_index=MRL_STATE_FIELD_INDEX,
    closed_value=MRL_CLOSED_VALUE,
    open_value=MRL_OPEN_VALUE,
)
print(f'{len(records)} ảnh; {len({item.subject_id for item in records})} subjects')
print(f'Mắt mở: {sum(item.label == 0 for item in records)}; mắt nhắm: {sum(item.label == 1 for item in records)}')
train_records, validation_records, test_records = split_by_subject(records, seed=42)
for name, split in [('train', train_records), ('validation', validation_records), ('test', test_records)]:
    print(f'{name}: {len(split)} ảnh, {len({item.subject_id for item in split})} subjects')
assert not ({x.subject_id for x in train_records} & {x.subject_id for x in validation_records})
assert not ({x.subject_id for x in train_records} & {x.subject_id for x in test_records})
assert not ({x.subject_id for x in validation_records} & {x.subject_id for x in test_records})

In [ ]:
IMAGE_SIZE = (96, 96)
BATCH_SIZE = 32
EPOCHS_PER_STAGE = 20
train_ds = make_tf_dataset(train_records, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, training=True)
validation_ds = make_tf_dataset(validation_records, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, training=False)
test_ds = make_tf_dataset(test_records, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, training=False)

model = train_mobilenet_binary(
    train_ds, validation_ds, image_size=IMAGE_SIZE, epochs_per_stage=EPOCHS_PER_STAGE
)
confusion_matrix, roc_auc = report_binary_metrics(model, test_ds)
print('Test ROC-AUC (đo trên dataset đang dùng):', roc_auc)
export_tflite(model, PROJECT_ROOT / 'models' / 'eye_classifier.tflite')